In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    for block in message.content:
        if block.type == "text":
            return block.text

## 課題: 構造化出力で、評価用のテストデータセット(`dataset.json`)を作る

- 対応: L11「テストデータセットの生成」。ページのコードは、プリフィルとストップシーケンスを使うが、現行モデルでは使えない可能性が高い(L08)。代わりに、構造化出力(`output_config.format`)を使う。
- 今回、新しく覚えること: **複数件のデータ**を、スキーマで表す(配列の中に、オブジェクトが並ぶ形)。
- 目標: AWS 関連のコーディング課題(Python、JSON、正規表現)を3件、`{"task": "..."}` の形で生成し、`dataset.json` に保存する。
- スキーマの設計について(私の判断): ページの出力は、配列が一番外側にある形だった。ここでは、一番外側を `object` にして、その中の `tasks` というキーに配列を入れる形にする。構造化出力のルートに配列を使えるかは、私は確認していない(確かでない)ため。
- `chat()` は `output_config` を受け取らないので、`client.messages.create()` を直接呼ぶ。
- API を呼ぶので、少額の料金がかかる。モデルは `claude-haiku-5-5`(ページの「高速なモデルで足りる」に合わせる)。
- ヒントは、下の「ヒント」のセルにある。まず、見ずに、試してみる。

In [4]:
# TODO 1: JSON スキーマを書く
#   - 全体は object で、キーは "tasks" の1つ。
#   - "tasks" は配列。配列の各要素は object で、キーは "task" の1つ(文字列)。
#   - 構造:  {"tasks": [ {"task": "..."}, {"task": "..."}, ... ]}
schema = {
    "type": "object",
    "properties": {
        "tasks": {
            "type": "array",  # TODO: 配列を表す type
            "items": {
                "type": "object",  # TODO: 配列の1要素の type
                "properties": {
                    "task": {"type": "string"},
                },
                "required": ["task"],  # TODO: 必須にするキー
                "additionalProperties": False,
            },
        },
    },
    "required": ["tasks"],
    "additionalProperties": False,
}

In [14]:
import json

# 構造化出力の応答から、本文を取り出す関数(ex06 で作ったものと同じ)
def get_text_from_response(response):
    for block in response.content:
        if block.type == "text":
            return block.text
    return None


def generate_dataset():
    prompt = """
プロンプト評価用の評価データセットを生成してください。このデータセットは、AWS に関連するタスクについて、Python、JSON、または正規表現を生成するプロンプトを評価するために使います。各項目は、Python、JSON、または正規表現を書くことで解決できるタスクです。

* 単一の Python 関数、単一の JSON オブジェクト、または単一の正規表現を書くことで解決できるタスクに絞ってください
* 書くコードの量が多くないタスクに絞ってください

Python・JSON・正規表現を各1件ずつを必ず作成してください。（全3件）
"""
    messages = []
    add_user_message(messages, prompt)

    # TODO 2: create() に output_config を渡して呼び出す(ex06 と同じ3段の形。schema は TODO 1 の変数)
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        output_config={"format": {"type": "json_schema", "schema": schema}}
        # TODO: output_config=...
    )

    print("stop_reason:", response.stop_reason)

    # TODO 3: 本文を取り出して、json.loads() で辞書にし、"tasks" の配列だけを返す
    text = get_text_from_response(response)
    data = json.loads(text)  # TODO: json.loads(text)
    return data["tasks"]  # TODO: data の "tasks" を返す

In [21]:
# TODO 4: 関数を呼び出して、結果を表示する。件数も確認する
dataset = generate_dataset()
print(dataset)
print(len(dataset))  # 3 件になっているか

counter = 0

while len(dataset) != 3:
    print("3件ではないため、再度生成します。")
    dataset = generate_dataset()
    print(dataset)
    print(len(dataset))
    counter += 1
    if counter >= 4:
        print("5回試したが、3件にならなかった。")
        raise Exception("3件にならなかった。")

print(counter+1, "回で3件になった。")

stop_reason: end_turn
[{'task': 'S3バケット名が命名規則（小文字・数字・ハイフンのみ、3〜63文字、先頭と末尾は英数字）を満たしているかを判定する単一のPython関数 is_valid_s3_bucket_name(name: str) -> bool を書いてください。'}, {'task': "AWS Lambda関数のIAMロールに付与する最小権限のポリシーを表す単一のJSONオブジェクトを作成してください。内容は、指定したS3バケット 'my-app-logs' の 'logs/' プレフィックス配下へ s3:PutObject のみを許可するIAMポリシードキュメント（Version 2012-10-17）です。"}, {'task': 'CloudWatch Logsのログ行から、AWS EC2インスタンスID（i-で始まり、8桁または17桁の16進数が続く形式）を抽出する単一の正規表現を書いてください。'}]
3
1 回で3件になった。


In [22]:
# TODO 5: dataset を、dataset.json に保存する
#   - open() の第2引数は、書き込みを表す文字列。
#   - 日本語やそのままの文字を保つため、encoding と ensure_ascii を指定している。
with open("dataset.json", "w", encoding="utf-8") as f:  # TODO: 書き込みモード
    json.dump(dataset, f, indent=2, ensure_ascii=False)

# 保存できたか、読み戻して確認する
with open("dataset.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded)

[{'task': 'S3バケット名が命名規則（小文字・数字・ハイフンのみ、3〜63文字、先頭と末尾は英数字）を満たしているかを判定する単一のPython関数 is_valid_s3_bucket_name(name: str) -> bool を書いてください。'}, {'task': "AWS Lambda関数のIAMロールに付与する最小権限のポリシーを表す単一のJSONオブジェクトを作成してください。内容は、指定したS3バケット 'my-app-logs' の 'logs/' プレフィックス配下へ s3:PutObject のみを許可するIAMポリシードキュメント（Version 2012-10-17）です。"}, {'task': 'CloudWatch Logsのログ行から、AWS EC2インスタンスID（i-で始まり、8桁または17桁の16進数が続く形式）を抽出する単一の正規表現を書いてください。'}]


## ヒント(困ったときだけ開く)

**TODO 1(スキーマ)**
- 「配列」の型名は `"array"`、「オブジェクト」の型名は `"object"`。
- 配列の中身は `"items"` に書く(ex06 で書いた形と同じ)。
- `required` は、キーの名前を文字列で並べたリスト。ここでは `task` を必須にしたい。
- `additionalProperties` は、オブジェクトごとに書く必要がある(ここでは外側と内側の2か所)。

**TODO 2(`output_config`)**
- 形は、ex06 と同じ `{"format": {"type": "json_schema", "schema": ...}}` の3段。

**TODO 3(取り出し)**
- `json.loads(text)` の結果は、辞書。辞書からキーで値を取り出す書き方は、`data["tasks"]`。
- `text` が `None` のときは、`json.loads` がエラーになる。`stop_reason` の表示を見て、`end_turn` かを確認する。

**TODO 4(件数)**
- `len(dataset)` は、リストの要素数を返す。「3件」と頼んだのに、違う数が返ったら、それはなぜか考えてみる(構造化出力が保証するのは、形であって、件数ではない。私の理解で、公式での確認はしていない)。

**TODO 5(保存)**
- 書き込みモードの文字列は、`"w"`(write)。
- 保存先は、ノートブックと同じフォルダ。

**エラーが出たら**
- 最後の行を読む。400 の場合、`message` の文面に、どの項目が悪いかが書かれていることが多い。
- `KeyError` は、辞書に、そのキーがないという意味。

## 振り返り(自分の言葉で書く)

1. 生成された3件の `task` を読んで、**多様性**(似たタスクばかりでないか)と**現実性**(実際のユーザーが頼みそうか)を、自分の目で評価する。気になる点があれば、書く。
2. 一番外側を `object` にして、その中に配列を入れる形にした。ページの形(配列が一番外側)と比べて、後でデータセットを使う側(評価の実行)で、何か違いが出そうか。
3. 「3件」と頼んだとき、構造化出力は、件数を保証するか。確認した結果を書く。
4. L11 の確認問題で答えた「高速なモデルを使うことで、何を捨てているか」を、今回の出力で、実際に確かめられたか。